# Davinci — PyTorch Decoder Implementation & Smoke Tests

This notebook contains a standalone historical implementation of a small PyTorch decoder. The canonical maintained decoder now lives in `src/davinci/model.py`; Notebook 6 imports that canonical module directly from the repository. Use this notebook for its own short-sequence teaching/smoke-test cells, not as the source of truth for the current model.

**Important engineering boundary:** the reference attention implementation below builds a dense attention mask. It is suitable for correctness tests at short sequence lengths, not efficient 500K training. Long-context deployment requires a verified block-sparse/sliding-window kernel and a runtime that evicts local KV cache correctly. The full ~509M model is not instantiated by default.

## 1. Install and import

## 1. Check the Colab PyTorch runtime

Colab normally includes PyTorch. Avoid upgrading it blindly because that can replace CUDA-compatible wheels and break the GPU runtime. If PyTorch is missing, install a build matched to the runtime's CUDA version using the official PyTorch selector.

In [ ]:
import json, math, os, random
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED=42
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("torch:",torch.__version__,"device:",DEVICE)

## 2. Load the architecture configuration

The config below mirrors Notebook 4. Update `vocab_size` to the actual frozen tokenizer size before training. Keep a separate small config for smoke tests so the notebook does not accidentally allocate hundreds of millions of parameters.

In [ ]:
TARGET_CONFIG = {
 "vocab_size":32000,"hidden_size":1280,"num_hidden_layers":28,
 "num_attention_heads":20,"num_key_value_heads":4,"head_dim":64,
 "intermediate_size":3328,"norm_eps":1e-5,"rope_theta":10000.0,
 "max_position_embeddings":4096,"local_window":4096,
 "global_every_n_layers":4,"tie_word_embeddings":True,
 "dropout":0.0
}
SMOKE_CONFIG = {
 **TARGET_CONFIG,"vocab_size":512,"hidden_size":128,
 "num_hidden_layers":4,"num_attention_heads":4,
 "num_key_value_heads":2,"head_dim":32,"intermediate_size":352,
 "max_position_embeddings":256,"local_window":64,
 "global_every_n_layers":2
}
def validate_config(c):
    assert c["hidden_size"]==c["num_attention_heads"]*c["head_dim"]
    assert c["num_attention_heads"]%c["num_key_value_heads"]==0
    assert c["head_dim"]%2==0
    assert c["num_hidden_layers"]>0 and c["vocab_size"]>0
    assert c["global_every_n_layers"]>0 and c["local_window"]>0
validate_config(TARGET_CONFIG)
validate_config(SMOKE_CONFIG)
print("Target and smoke configurations validated.")

## 3. Implement RMSNorm, RoPE, GQA attention, and SwiGLU

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight=nn.Parameter(torch.ones(dim))
        self.eps=eps
    def forward(self,x):
        x_dtype=x.dtype
        y=x.float()*torch.rsqrt(x.float().pow(2).mean(dim=-1,keepdim=True)+self.eps)
        return (y*self.weight.float()).to(x_dtype)

def apply_rope(q,k,theta=10000.0):
    # q: [B,H,T,D], k: [B,KVH,T,D]
    d=q.size(-1)
    inv_freq=1.0/(theta**(torch.arange(0,d,2,device=q.device,dtype=torch.float32)/d))
    positions=torch.arange(q.size(-2),device=q.device,dtype=torch.float32)
    freqs=torch.outer(positions,inv_freq)
    cos=freqs.cos().to(q.dtype)[None,None,:,:]
    sin=freqs.sin().to(q.dtype)[None,None,:,:]
    def rotate(x):
        x1=x[...,0::2]; x2=x[...,1::2]
        return torch.stack((x1*cos-x2*sin,x1*sin+x2*cos),dim=-1).flatten(-2)
    return rotate(q),rotate(k)

def repeat_kv(x,groups):
    # [B,KVH,T,D] -> [B,H,T,D]
    if groups==1: return x
    b,kvh,t,d=x.shape
    return x[:,:,None,:,:].expand(b,kvh,groups,t,d).reshape(b,kvh*groups,t,d)

def make_attention_mask(seq_len,window,global_layer,device):
    # SDPA boolean masks: True means query may attend to key.
    q=torch.arange(seq_len,device=device)[:,None]
    k=torch.arange(seq_len,device=device)[None,:]
    causal=(k<=q)
    if global_layer: return causal
    return causal & ((q-k)<window)

class GQAAttention(nn.Module):
    def __init__(self,c,layer_idx):
        super().__init__()
        d=c["hidden_size"]; h=c["num_attention_heads"]; kv=c["num_key_value_heads"]; hd=c["head_dim"]
        self.heads=h; self.kv_heads=kv; self.groups=h//kv; self.head_dim=hd
        self.window=c["local_window"]; self.theta=c["rope_theta"]
        self.global_layer=((layer_idx+1)%c["global_every_n_layers"]==0)
        self.q_proj=nn.Linear(d,h*hd,bias=False)
        self.k_proj=nn.Linear(d,kv*hd,bias=False)
        self.v_proj=nn.Linear(d,kv*hd,bias=False)
        self.o_proj=nn.Linear(h*hd,d,bias=False)
        self.dropout=c["dropout"]
    def forward(self,x):
        b,t,_=x.shape
        q=self.q_proj(x).view(b,t,self.heads,self.head_dim).transpose(1,2)
        k=self.k_proj(x).view(b,t,self.kv_heads,self.head_dim).transpose(1,2)
        v=self.v_proj(x).view(b,t,self.kv_heads,self.head_dim).transpose(1,2)
        q,k=apply_rope(q,k,self.theta)
        k=repeat_kv(k,self.groups); v=repeat_kv(v,self.groups)
        mask=make_attention_mask(t,self.window,self.global_layer,x.device)
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,
              dropout_p=self.dropout if self.training else 0.0,is_causal=False)
        y=y.transpose(1,2).contiguous().view(b,t,-1)
        return self.o_proj(y)

class SwiGLU(nn.Module):
    def __init__(self,dim,hidden):
        super().__init__()
        self.gate=nn.Linear(dim,hidden,bias=False)
        self.up=nn.Linear(dim,hidden,bias=False)
        self.down=nn.Linear(hidden,dim,bias=False)
    def forward(self,x):
        return self.down(F.silu(self.gate(x))*self.up(x))

## 4. Decoder block and causal language model

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self,c,layer_idx):
        super().__init__()
        d=c["hidden_size"]
        self.attn_norm=RMSNorm(d,c["norm_eps"])
        self.attn=GQAAttention(c,layer_idx)
        self.mlp_norm=RMSNorm(d,c["norm_eps"])
        self.mlp=SwiGLU(d,c["intermediate_size"])
    def forward(self,x):
        x=x+self.attn(self.attn_norm(x))
        x=x+self.mlp(self.mlp_norm(x))
        return x

class DavinciForCausalLM(nn.Module):
    def __init__(self,c):
        super().__init__()
        validate_config(c)
        self.config=dict(c)
        self.embed_tokens=nn.Embedding(c["vocab_size"],c["hidden_size"])
        self.layers=nn.ModuleList([DecoderBlock(c,i) for i in range(c["num_hidden_layers"])])
        self.final_norm=RMSNorm(c["hidden_size"],c["norm_eps"])
        self.lm_head=nn.Linear(c["hidden_size"],c["vocab_size"],bias=False)
        if c["tie_word_embeddings"]:
            self.lm_head.weight=self.embed_tokens.weight
        self.apply(self._init_weights)
    def _init_weights(self,module):
        if isinstance(module,nn.Linear):
            nn.init.normal_(module.weight,mean=0.0,std=0.02)
        elif isinstance(module,nn.Embedding):
            nn.init.normal_(module.weight,mean=0.0,std=0.02)
    def forward(self,input_ids,labels=None):
        x=self.embed_tokens(input_ids)
        for layer in self.layers: x=layer(x)
        logits=self.lm_head(self.final_norm(x))
        loss=None
        if labels is not None:
            shift_logits=logits[:,:-1,:].contiguous()
            shift_labels=labels[:,1:].contiguous()
            loss=F.cross_entropy(shift_logits.view(-1,shift_logits.size(-1)),
                                 shift_labels.view(-1),ignore_index=-100)
        return {"logits":logits,"loss":loss}

## 5. Parameter-count parity test

The analytical formula from the architecture notebook is checked against the actual PyTorch module. This catches accidental biases, untied embeddings, or dimension mismatches.

In [ ]:
def analytical_parameter_count(c):
    d=c["hidden_size"]; h=c["num_attention_heads"]; kv=c["num_key_value_heads"]
    hd=c["head_dim"]; f=c["intermediate_size"]; L=c["num_hidden_layers"]; V=c["vocab_size"]
    attn=2*d*d+2*kv*hd*d
    mlp=3*d*f
    norms=2*d
    return L*(attn+mlp+norms)+V*d+d+(0 if c["tie_word_embeddings"] else V*d)

model=DavinciForCausalLM(SMOKE_CONFIG)
actual=sum(p.numel() for p in model.parameters())
expected=analytical_parameter_count(SMOKE_CONFIG)
print({"actual":actual,"analytical":expected,"match":actual==expected})
assert actual==expected
print("Smoke model parameters:",f"{actual:,}")

## 6. Forward, loss, and backward smoke test

In [ ]:
model=model.to(DEVICE)
batch=2; seq=48
input_ids=torch.randint(0,SMOKE_CONFIG["vocab_size"],(batch,seq),device=DEVICE)
labels=input_ids.clone()
out=model(input_ids,labels=labels)
assert out["logits"].shape==(batch,seq,SMOKE_CONFIG["vocab_size"])
assert out["loss"] is not None and torch.isfinite(out["loss"])
out["loss"].backward()
grad_ok=all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
assert grad_ok
print({"logits_shape":tuple(out["logits"].shape),"loss":float(out["loss"].detach()),"finite_gradients":bool(grad_ok)})

## 7. Causality and local-window behavior checks

Changing future tokens must not alter earlier-position logits in evaluation mode. Local attention must mask keys older than the configured window; global layers must permit all prior keys. These tests verify mask semantics, not long-context kernel efficiency.

In [ ]:
model.eval()
with torch.no_grad():
    a=torch.randint(0,SMOKE_CONFIG["vocab_size"],(1,32),device=DEVICE)
    b=a.clone(); b[:,20:]=torch.randint(0,SMOKE_CONFIG["vocab_size"],(1,12),device=DEVICE)
    la=model(a)["logits"]; lb=model(b)["logits"]
    causal_error=(la[:,:20]-lb[:,:20]).abs().max().item()
assert causal_error<1e-5,causal_error
local=make_attention_mask(10,4,False,DEVICE)
global_mask=make_attention_mask(10,4,True,DEVICE)
assert not local[9,0] and local[9,6] and local[9,9]
assert global_mask[9,0] and not global_mask[0,1]
print({"future_token_invariance_max_error":causal_error,"local_and_global_masks":"passed"})

## 8. Save and reload a checkpoint

In [ ]:
CKPT=Path("/content/davinci_smoke_checkpoint.pt")
torch.save({"config":SMOKE_CONFIG,"model_state_dict":model.state_dict()},CKPT)
loaded=torch.load(CKPT,map_location=DEVICE,weights_only=True)
reloaded=DavinciForCausalLM(loaded["config"]).to(DEVICE)
reloaded.load_state_dict(loaded["model_state_dict"])
reloaded.eval()
with torch.no_grad():
    reload_error=(model(input_ids)["logits"]-reloaded(input_ids)["logits"]).abs().max().item()
assert reload_error<1e-6,reload_error
print({"checkpoint":str(CKPT),"reload_max_error":reload_error})

## 9. Optional instantiate the full target model

This may require several GB of RAM/VRAM even before training state and activations. Do not run on a GPU unless you have checked available memory. Use CPU only if system RAM is sufficient. This cell is optional and does not start training.

In [ ]:
INSTANTIATE_FULL_TARGET=False
if INSTANTIATE_FULL_TARGET:
    target=DavinciForCausalLM(TARGET_CONFIG)
    n=sum(p.numel() for p in target.parameters())
    print("Instantiated parameter count:",n,"(millions:",n/1e6,")")
    assert n==analytical_parameter_count(TARGET_CONFIG)

## 10. Export implementation config and smoke-test report

In [ ]:
OUT=Path("/content/davinci_model_implementation")
OUT.mkdir(parents=True,exist_ok=True)
(OUT/"davinci_model_config.json").write_text(json.dumps(TARGET_CONFIG,indent=2),encoding="utf-8")
report={
 "smoke_config_parameters":actual,
 "analytical_smoke_parameters":expected,
 "parameter_count_match":actual==expected,
 "forward_loss_backward":"passed",
 "causality_test":"passed",
 "local_global_mask_test":"passed",
 "checkpoint_reload_max_logit_error":reload_error,
 "target_config_analytical_parameters":analytical_parameter_count(TARGET_CONFIG),
 "attention_implementation_warning":"Dense boolean masks are correctness reference only; not suitable for 500K training."
}
(OUT/"implementation_smoke_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
print(report)

## 11. Download artifacts

In [ ]:
from google.colab import files
files.download(str(OUT/"davinci_model_config.json"))
files.download(str(OUT/"implementation_smoke_report.json"))

## 12. Limitations and production requirements

- This is a reference implementation for correctness and experimentation, not a production-optimized transformer.
- The dense mask scales quadratically in sequence length. Replace it with a tested block-sparse/local attention kernel before long-context training.
- This code recomputes the full prefix and does not implement incremental KV-cache decoding.
- RoPE here is standard fixed-frequency RoPE; 500K context requires a separately designed, validated positional-extension strategy.
- No tokenizer-specific chat serialization is included. Keep the tokenizer special-token IDs and model config synchronized.
- No distributed training, optimizer, scheduler, gradient checkpointing, packing, or data loader is included yet.
- Before full training, compare outputs against an independent reference implementation and run mixed-precision, memory, throughput, and overfit-a-small-batch tests.

## 13. Export the model implementation as a reusable Python module

This export is retained for standalone experimentation. The current Notebook 6 training engine imports the canonical implementation from `src/davinci/` instead.

In [ ]:
import inspect
_export_objects=[RMSNorm,apply_rope,repeat_kv,make_attention_mask,GQAAttention,SwiGLU,DecoderBlock,DavinciForCausalLM]
_export_source="import torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\n"+"\\n\\n".join(inspect.getsource(obj) for obj in _export_objects)
_export_path=Path("/content/davinci_model.py")
_export_path.write_text(_export_source,encoding="utf-8")
print("Exported:",_export_path,"bytes:",_export_path.stat().st_size)
